What we are trying to do: Given the current game context, which lineup or substitution should a coach use?

Example contexts:
- we need points
- we need defense
- we are protecting a lead
- we are down late in the 4th
- a starter is resting
- opponent has a certain lineup on the floor

For now think of it as this: 
Based on historical lineup performance, recommend the best lineup for a given team and situation

# Initial EDA

In [1]:
import pandas as pd

path = "/Users/manishkavuri/Desktop/nba-lineup-decision-engine/data/processed/final/team_lineup_stint_features_v1_2024_25.parquet"

df = pd.read_parquet(path)

print(df.shape)
print(df["game_id"].nunique())
print(df["lineup_key"].nunique())

display(df.head())

(81536, 36)
1230
3274


,stint_id,game_id,stint_number,period,is_overtime,start_clock,end_clock,start_seconds_remaining,end_seconds_remaining,duration_seconds,...,lineup_key,opponent_lineup_key,score_margin_start,score_margin_end,score_margin_change,points_for_per_min,points_against_per_min,net_points_per_min,net_points_per_48,is_zero_duration
0,0022400001_001,0022400001,1,1,False,PT12M00.00S,PT07M02.00S,720.0,422.0,298.0,...,201143-201950-1627759-1628369-1628401,203991-1630552-1630700-1630811-1642258,0,-3,-3,1.610738,2.214765,-0.604027,-28.993289,False
1,0022400001_002,0022400001,2,1,False,PT07M02.00S,PT05M57.00S,422.0,357.0,65.0,...,201143-201950-1627759-1628369-1628401,203991-1630552-1630700-1630811-1642258,-3,1,4,3.692308,0.000000,3.692308,177.230769,False
2,0022400001_003,0022400001,3,1,False,PT05M57.00S,PT04M51.00S,357.0,291.0,66.0,...,201143-201950-1627759-1628369-1628401,203991-1630552-1630700-1630811-1642258,1,1,0,1.818182,1.818182,0.000000,0.000000,False
3,0022400001_004,0022400001,4,1,False,PT04M51.00S,PT04M27.00S,291.0,267.0,24.0,...,201143-201950-1627759-1628369-1628401,203991-1630552-1630700-1630811-1642258,1,-2,-3,0.000000,7.500000,-7.500000,-360.000000,False
4,0022400001_005,0022400001,5,1,False,PT04M27.00S,PT00M26.30S,267.0,26.3,240.7,...,201143-201950-1627759-1628369-1628401,203991-1630552-1630700-1630811-1642258,-2,-1,1,3.489821,3.240548,0.249273,11.965102,False


In [2]:
# filter out the zero duration rows 
df_nonzero = df[df["duration_seconds"] > 0].copy()

print(df_nonzero.shape)
print(df_nonzero["game_id"].nunique())
print(df_nonzero["lineup_key"].nunique())

(74326, 36)
1230
3274


In [3]:
# let's also create a min 30 second stint filter
df_30 = df[df["duration_seconds"] >= 30].copy()

print(df_30.shape)
print(df_30["game_id"].nunique())
print(df_30["lineup_key"].nunique())

(60678, 36)
1230
3274


## Dataset health

In [5]:
df_30["duration_seconds"].describe()

count    60678.000000
mean       113.974712
std         77.475415
min         30.000000
25%         58.700000
50%         91.000000
75%        143.475000
max        720.000000
Name: duration_seconds, dtype: float64

In [6]:
df_30["period"].value_counts().sort_index()

period
1    13370
2    17220
3    14224
4    15614
5      236
6       14
Name: count, dtype: int64

In [14]:
display(df_30["net_points"].describe())
display(df_30["net_points_per_48"].describe())

count    60678.000000
mean         0.000000
std          3.577318
min        -22.000000
25%         -2.000000
50%          0.000000
75%          2.000000
max         22.000000
Name: net_points, dtype: float64

count    60678.000000
mean         0.000000
std        116.327437
min       -576.000000
25%        -69.266455
50%          0.000000
75%         69.266455
max        576.000000
Name: net_points_per_48, dtype: float64

## Lineup Usage

In [9]:
lineup_usage = (
    df_nonzero
    .groupby(["team_id", "lineup_key"])
    .agg(
        total_seconds=("duration_seconds", "sum"),
        total_minutes=("duration_minutes", "sum"),
        stints=("stint_id", "count"),
        games=("game_id", "nunique"),
        net_points=("net_points", "sum"),
        points_for=("points_for", "sum"),
        points_against=("points_against", "sum"),
    )
    .reset_index()
)

lineup_usage["net_points_per_48"] = (
    lineup_usage["net_points"] / lineup_usage["total_minutes"] * 48
)

lineup_usage = lineup_usage.sort_values("total_minutes", ascending=False)

display(lineup_usage.head(20))

,team_id,lineup_key,total_seconds,total_minutes,stints,games,net_points,points_for,points_against,net_points_per_48
1423,1610612750,201144-203497-203944-1628978-1629638,79200.0,1320.0,902,51,46,3084,3038,1.672727
1830,1610612754,204456-1626167-1627783-1628418-1629614,57900.0,965.0,654,38,22,2300,2278,1.094301
3084,1610612765,202699-203501-1627736-1630191-1630595,50400.0,840.0,598,32,-59,1907,1966,-3.371429
764,1610612744,201939-202710-203110-1626172-1627741,48240.0,804.0,555,27,245,2008,1763,14.626866
1586,1610612752,1626157-1626166-1628384-1628404-1628969,46800.0,780.0,519,45,138,1881,1743,8.492308
1431,1610612750,201144-203497-203944-1630162-1630183,40320.0,672.0,368,52,111,1575,1464,7.928571
1608,1610612752,1626157-1628384-1628404-1628969-1628973,36720.0,612.0,324,48,34,1438,1404,2.666667
971,1610612746,201587-201935-1626181-1627739-1627826,35280.0,588.0,401,29,-89,1285,1374,-7.265306
200,1610612739,1627747-1627777-1628378-1628386-1629636,33840.0,564.0,404,20,64,1373,1309,5.446809
3080,1610612765,202699-203471-203501-1627736-1630191,33120.0,552.0,401,18,63,1351,1288,5.478261


## Best and Worst Lineups

In [10]:
min_minutes = 20

qualified_lineups = lineup_usage[
    lineup_usage["total_minutes"] >= min_minutes
].copy()

best_lineups = qualified_lineups.sort_values(
    "net_points_per_48",
    ascending=False
)

worst_lineups = qualified_lineups.sort_values(
    "net_points_per_48",
    ascending=True
)

display(best_lineups.head(20))
display(worst_lineups.head(20))

,team_id,lineup_key,total_seconds,total_minutes,stints,games,net_points,points_for,points_against,net_points_per_48
1786,1610612753,203914-1628371-1629048-1630175-1630243,1440.0,24.0,16,1,33,80,47,66.000000
2442,1610612760,1627936-1628983-1629652-1630198-1631096,1440.0,24.0,16,2,31,63,32,62.000000
2502,1610612760,1629026-1630198-1630598-1631119-1641794,1440.0,24.0,13,2,31,84,53,62.000000
513,1610612742,202681-1629023-1629656-1630230-1641726,1440.0,24.0,11,2,31,68,37,62.000000
201,1610612739,1627747-1627777-1628378-1629636-1629660,1440.0,24.0,12,2,30,72,42,60.000000
1057,1610612747,1627827-1629029-1629637-1630559-1630692,2160.0,36.0,22,3,43,109,66,57.333333
648,1610612743,201566-1627750-1629008-1631124-1631212,1440.0,24.0,13,2,28,75,47,56.000000
1009,1610612746,201587-202695-203992-1627732-1627739,2880.0,48.0,29,4,48,136,88,48.000000
3086,1610612765,202699-203501-1627736-1630191-1631105,1440.0,24.0,15,2,24,69,45,48.000000
1614,1610612752,1626157-1628384-1628404-1629011-1629013,2880.0,48.0,21,4,47,126,79,47.000000


,team_id,lineup_key,total_seconds,total_minutes,stints,games,net_points,points_for,points_against,net_points_per_48
2915,1610612764,1628398-1629673-1641731-1642259-1642273,1440.0,24.0,12,2,-37,40,77,-74.000000
2566,1610612761,1629628-1630193-1630567-1641711-1642367,1440.0,24.0,14,2,-35,41,76,-70.000000
1938,1610612755,200768-202704-1626162-1627824-1630178,1440.0,24.0,11,2,-31,35,66,-62.000000
2992,1610612764,202685-1628398-1629673-1630557-1631098,2160.0,36.0,19,2,-44,55,99,-58.666667
711,1610612743,201566-203967-203999-1627750-1629008,1440.0,24.0,18,2,-28,48,76,-56.000000
1616,1610612752,1626157-1628384-1628404-1629013-1630173,1440.0,24.0,12,2,-28,39,67,-56.000000
3028,1610612764,203114-1629673-1630264-1630551-1641774,1440.0,24.0,12,1,-28,41,69,-56.000000
2939,1610612764,1630550-1630551-1641732-1641774-1642259,1440.0,24.0,17,1,-26,44,70,-52.000000
296,1610612740,203468-1627742-1627749-1629750-1630526,1440.0,24.0,19,2,-26,48,74,-52.000000
3213,1610612766,203552-203901-203994-1628970-1630163,1440.0,24.0,14,1,-26,45,71,-52.000000


### Who should be on the floor when we need points

In [11]:
qualified_lineups["points_for_per_48"] = (
    qualified_lineups["points_for"] / qualified_lineups["total_minutes"] * 48
)

best_offensive_lineups = qualified_lineups.sort_values(
    "points_for_per_48",
    ascending=False
)

display(best_offensive_lineups.head(20))

,team_id,lineup_key,total_seconds,total_minutes,stints,games,net_points,points_for,points_against,net_points_per_48,points_for_per_48
741,1610612743,203932-1627750-1629618-1630192-1631124,1440.0,24.0,16,2,23,87,64,46.000000,174.000000
2502,1610612760,1629026-1630198-1630598-1631119-1641794,1440.0,24.0,13,2,31,84,53,62.000000,168.000000
1786,1610612753,203914-1628371-1629048-1630175-1630243,1440.0,24.0,16,1,33,80,47,66.000000,160.000000
1268,1610612748,202710-1626179-1628389-1629130-1629639,1440.0,24.0,12,2,13,77,64,26.000000,154.000000
832,1610612744,203110-203937-1626172-1627741-1630228,1440.0,24.0,14,1,22,77,55,44.000000,154.000000
1853,1610612754,204456-1627783-1630169-1631097-1631245,1440.0,24.0,14,2,16,77,61,32.000000,154.000000
2877,1610612763,203935-1628991-1629634-1629723-1630583,1440.0,24.0,14,1,21,77,56,42.000000,154.000000
2450,1610612760,1627936-1629026-1630198-1630598-1631096,1440.0,24.0,15,1,22,76,54,44.000000,152.000000
2197,1610612757,1629680-1630166-1630625-1631133-1631200,1440.0,24.0,12,2,20,76,56,40.000000,152.000000
634,1610612742,203915-203939-1629023-1630230-1630539,1440.0,24.0,22,1,-4,76,80,-8.000000,152.000000


## Player EDA

In [12]:
def parse_lineup_key(lineup_key):
    return [int(x) for x in str(lineup_key).split("-") if x != ""]

In [13]:
player_rows = []

for _, row in df_nonzero.iterrows():
    player_ids = parse_lineup_key(row["lineup_key"])

    for pid in player_ids:
        player_rows.append({
            "game_id": row["game_id"],
            "team_id": row["team_id"],
            "player_id": pid,
            "duration_seconds": row["duration_seconds"],
            "duration_minutes": row["duration_minutes"],
            "net_points": row["net_points"],
            "points_for": row["points_for"],
            "points_against": row["points_against"],
        })

player_df = pd.DataFrame(player_rows)

player_summary = (
    player_df
    .groupby(["team_id", "player_id"])
    .agg(
        total_minutes=("duration_minutes", "sum"),
        net_points=("net_points", "sum"),
        points_for=("points_for", "sum"),
        points_against=("points_against", "sum"),
        stints=("game_id", "count"),
        games=("game_id", "nunique"),
    )
    .reset_index()
)

player_summary["net_points_per_48"] = (
    player_summary["net_points"] / player_summary["total_minutes"] * 48
)

player_summary["points_for_per_48"] = (
    player_summary["points_for"] / player_summary["total_minutes"] * 48
)

display(
    player_summary[player_summary["total_minutes"] >= 100]
    .sort_values("net_points_per_48", ascending=False)
    .head(20)
)

,team_id,player_id,total_minutes,net_points,points_for,points_against,stints,games,net_points_per_48,points_for_per_48
95,1610612741,1631207,144.0,95,407,312,79,9,31.666667,135.666667
459,1610612760,1631119,341.0,165,887,722,196,16,23.225806,124.856305
49,1610612739,1630171,341.0,164,915,751,177,26,23.085044,128.797654
461,1610612760,1641717,552.0,244,1435,1191,289,44,21.217391,124.782609
225,1610612748,1630528,144.0,57,365,308,65,11,19.000000,121.666667
457,1610612760,1631096,672.0,266,1741,1475,377,32,19.000000,124.357143
51,1610612739,1630596,852.0,300,2215,1915,428,71,16.901408,124.788732
258,1610612750,1628408,108.0,36,248,212,65,9,16.000000,110.222222
332,1610612754,1628396,291.0,93,750,657,204,14,15.340206,123.711340
31,1610612738,1630573,481.0,153,1198,1045,241,30,15.268191,119.550936


These rankings are useful for finding interesting lineups, but they should not be treated as final recommendations yet because many lineups have small sample sizes. We should compare results using higher thresholds like 50, 100, or 200 minutes, or use a shrinkage-adjusted rating.

Player-level EDA helps identify players associated with high-performing lineups, but it should not be interpreted as true individual value without controlling for teammates, opponents, and game context.

In [15]:
for min_minutes in [20, 50, 100, 200]:
    qualified = lineup_usage[lineup_usage["total_minutes"] >= min_minutes]
    print(min_minutes, qualified.shape[0])

20 1394
50 465
100 218
200 88


as we can see we lose a lot of lineup stints as we filter more and more, so this tells me I might need to pull more data from previous seasons as well.

In [16]:
qualified_lineups["points_against_per_48"] = (
    qualified_lineups["points_against"] / qualified_lineups["total_minutes"] * 48
)

best_defensive_lineups = qualified_lineups.sort_values(
    "points_against_per_48",
    ascending=True
)

Missing Values

In [17]:
df.isna().sum().sort_values(ascending=False).head(20)

net_points_per_48         7210
net_points_per_min        7210
points_against_per_min    7210
points_for_per_min        7210
stint_id                     0
ended_by_substitution        0
score_for_end                0
score_against_end            0
points_for                   0
points_against               0
net_points                   0
opponent_lineup_key          0
lineup_key                   0
game_id                      0
score_margin_start           0
score_margin_end             0
score_margin_change          0
score_against_start          0
score_for_start              0
opponent_lineup_names        0
dtype: int64

In [18]:
df["is_zero_duration"].value_counts(normalize=True)

is_zero_duration
False    0.911573
True     0.088427
Name: proportion, dtype: float64

In [19]:
for m in [20, 50, 100, 200, 500]:
    n = (lineup_usage["total_minutes"] >= m).sum()
    print(f"Lineups with at least {m} minutes: {n}")

Lineups with at least 20 minutes: 1394
Lineups with at least 50 minutes: 465
Lineups with at least 100 minutes: 218
Lineups with at least 200 minutes: 88
Lineups with at least 500 minutes: 14


In [20]:
qualified_lineups["points_against_per_48"] = (
    qualified_lineups["points_against"] / qualified_lineups["total_minutes"] * 48
)

display(
    qualified_lineups.sort_values("points_for_per_48", ascending=False).head(10)
)

display(
    qualified_lineups.sort_values("points_against_per_48", ascending=True).head(10)
)

display(
    qualified_lineups.sort_values("net_points_per_48", ascending=False).head(10)
)

,team_id,lineup_key,total_seconds,total_minutes,stints,games,net_points,points_for,points_against,net_points_per_48,points_for_per_48,points_against_per_48
741,1610612743,203932-1627750-1629618-1630192-1631124,1440.0,24.0,16,2,23,87,64,46.0,174.0,128.0
2502,1610612760,1629026-1630198-1630598-1631119-1641794,1440.0,24.0,13,2,31,84,53,62.0,168.0,106.0
1786,1610612753,203914-1628371-1629048-1630175-1630243,1440.0,24.0,16,1,33,80,47,66.0,160.0,94.0
1268,1610612748,202710-1626179-1628389-1629130-1629639,1440.0,24.0,12,2,13,77,64,26.0,154.0,128.0
832,1610612744,203110-203937-1626172-1627741-1630228,1440.0,24.0,14,1,22,77,55,44.0,154.0,110.0
1853,1610612754,204456-1627783-1630169-1631097-1631245,1440.0,24.0,14,2,16,77,61,32.0,154.0,122.0
2877,1610612763,203935-1628991-1629634-1629723-1630583,1440.0,24.0,14,1,21,77,56,42.0,154.0,112.0
2450,1610612760,1627936-1629026-1630198-1630598-1631096,1440.0,24.0,15,1,22,76,54,44.0,152.0,108.0
2197,1610612757,1629680-1630166-1630625-1631133-1631200,1440.0,24.0,12,2,20,76,56,40.0,152.0,112.0
634,1610612742,203915-203939-1629023-1630230-1630539,1440.0,24.0,22,1,-4,76,80,-8.0,152.0,160.0


,team_id,lineup_key,total_seconds,total_minutes,stints,games,net_points,points_for,points_against,net_points_per_48,points_for_per_48,points_against_per_48
2442,1610612760,1627936-1628983-1629652-1630198-1631096,1440.0,24.0,16,2,31,63,32,62.000000,126.000000,64.000000
1731,1610612753,203484-1628371-1629021-1629048-1630175,2160.0,36.0,28,2,8,63,55,10.666667,84.000000,73.333333
513,1610612742,202681-1629023-1629656-1630230-1641726,1440.0,24.0,11,2,31,68,37,62.000000,136.000000,74.000000
1212,1610612748,201567-202692-202710-1626196-1628389,2160.0,36.0,21,1,25,81,56,33.333333,108.000000,74.666667
1707,1610612753,202709-203484-203914-1628371-1629021,2160.0,36.0,23,3,18,74,56,24.000000,98.666667,74.666667
11,1610612737,1627747-1627777-1629027-1629611-1630249,2160.0,36.0,25,3,16,74,58,21.333333,98.666667,77.333333
3155,1610612766,201959-1628998-1629684-1630163-1630182,1440.0,24.0,18,1,20,59,39,40.000000,118.000000,78.000000
446,1610612741,202696-1628366-1629632-1630581-1641824,1440.0,24.0,13,2,12,51,39,24.000000,102.000000,78.000000
6,1610612737,1626204-1629027-1630168-1630249-1630700,2160.0,36.0,20,2,28,87,59,37.333333,116.000000,78.666667
872,1610612745,1627832-1630224-1630578-1631095-1641708,2160.0,36.0,21,3,32,91,59,42.666667,121.333333,78.666667


,team_id,lineup_key,total_seconds,total_minutes,stints,games,net_points,points_for,points_against,net_points_per_48,points_for_per_48,points_against_per_48
1786,1610612753,203914-1628371-1629048-1630175-1630243,1440.0,24.0,16,1,33,80,47,66.000000,160.000000,94.0
2442,1610612760,1627936-1628983-1629652-1630198-1631096,1440.0,24.0,16,2,31,63,32,62.000000,126.000000,64.0
2502,1610612760,1629026-1630198-1630598-1631119-1641794,1440.0,24.0,13,2,31,84,53,62.000000,168.000000,106.0
513,1610612742,202681-1629023-1629656-1630230-1641726,1440.0,24.0,11,2,31,68,37,62.000000,136.000000,74.0
201,1610612739,1627747-1627777-1628378-1629636-1629660,1440.0,24.0,12,2,30,72,42,60.000000,144.000000,84.0
1057,1610612747,1627827-1629029-1629637-1630559-1630692,2160.0,36.0,22,3,43,109,66,57.333333,145.333333,88.0
648,1610612743,201566-1627750-1629008-1631124-1631212,1440.0,24.0,13,2,28,75,47,56.000000,150.000000,94.0
1009,1610612746,201587-202695-203992-1627732-1627739,2880.0,48.0,29,4,48,136,88,48.000000,136.000000,88.0
3086,1610612765,202699-203501-1627736-1630191-1631105,1440.0,24.0,15,2,24,69,45,48.000000,138.000000,90.0
1614,1610612752,1626157-1628384-1628404-1629011-1629013,2880.0,48.0,21,4,47,126,79,47.000000,126.000000,79.0


The initial EDA confirms that the lineup dataset is usable across all 1,230 games. After removing zero-duration rows and filtering to stints of at least 30 seconds, we still retain over 60,000 team-stint observations. Most stints are short, with a median around 91 seconds, so lineup performance is noisy at the stint level. Because of that, the most useful analysis is aggregated by team and lineup. The first lineup rankings show which lineups have the best net scoring and offensive scoring rates, but these need minutes thresholds and shrinkage before being used as actual recommendations. Player-level summaries are also useful, but they should be interpreted as on-court association rather than isolated player impact.

Create Modeling Ready Dataset

In [21]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("/Users/manishkavuri/Desktop/nba-lineup-decision-engine")

df = pd.read_parquet(
    PROJECT_ROOT / "data/processed/final/team_lineup_stint_features_v1_2024_25.parquet"
)

model_df = df[df["duration_seconds"] >= 30].copy()

output_path = PROJECT_ROOT / "data/processed/final/modeling_lineup_stints_v1_2024_25.parquet"
model_df.to_parquet(output_path, index=False)

print(model_df.shape)
print(model_df["game_id"].nunique())
print(model_df["lineup_key"].nunique())
print("Saved:", output_path)

(60678, 36)
1230
3274
Saved: /Users/manishkavuri/Desktop/nba-lineup-decision-engine/data/processed/final/modeling_lineup_stints_v1_2024_25.parquet


Add lineup-level summary tables

In [22]:
lineup_summary = (
    model_df
    .groupby(["team_id", "lineup_key"])
    .agg(
        total_seconds=("duration_seconds", "sum"),
        total_minutes=("duration_minutes", "sum"),
        stints=("stint_id", "count"),
        games=("game_id", "nunique"),
        points_for=("points_for", "sum"),
        points_against=("points_against", "sum"),
        net_points=("net_points", "sum"),
    )
    .reset_index()
)

lineup_summary["points_for_per_48"] = (
    lineup_summary["points_for"] / lineup_summary["total_minutes"] * 48
)

lineup_summary["points_against_per_48"] = (
    lineup_summary["points_against"] / lineup_summary["total_minutes"] * 48
)

lineup_summary["net_points_per_48"] = (
    lineup_summary["net_points"] / lineup_summary["total_minutes"] * 48
)

lineup_summary_path = PROJECT_ROOT / "data/processed/final/lineup_summary_v1_2024_25.parquet"
lineup_summary.to_parquet(lineup_summary_path, index=False)

print(lineup_summary.shape)
print("Saved:", lineup_summary_path)

(3274, 12)
Saved: /Users/manishkavuri/Desktop/nba-lineup-decision-engine/data/processed/final/lineup_summary_v1_2024_25.parquet


Baseline Recommender:
need points → rank by points_for_per_48
Need stops → rank by lowest points_against_per_48
Balanced → rank by net_points_per_48

In [23]:
qualified = lineup_summary[lineup_summary["total_minutes"] >= 50].copy()

best_offense = qualified.sort_values("points_for_per_48", ascending=False)
best_defense = qualified.sort_values("points_against_per_48", ascending=True)
best_balanced = qualified.sort_values("net_points_per_48", ascending=False)

display(best_offense.head(20))
display(best_defense.head(20))
display(best_balanced.head(20))

,team_id,lineup_key,total_seconds,total_minutes,stints,games,points_for,points_against,net_points,points_for_per_48,points_against_per_48,net_points_per_48
647,1610612743,201566-1627750-1629008-1631124-1631128,3534.1,58.901667,27,5,177,143,34,144.240401,116.533205,27.707196
2437,1610612760,1627936-1628983-1629026-1629652-1630198,3537.4,58.956667,30,4,171,164,7,139.220897,133.521796,5.699101
2459,1610612760,1628392-1628983-1629026-1629652-1630598,4187.6,69.793333,36,6,202,155,47,138.924444,106.600439,32.324004
1007,1610612746,201587-202695-203992-1626181-1627732,4243.0,70.716667,31,6,204,144,60,138.468065,97.742164,40.725901
2264,1610612758,201942-1627734-1628368-1628370-1628989,4153.8,69.230000,37,4,199,164,35,137.974866,113.707930,24.266936
2430,1610612760,1627936-1628392-1629026-1630198-1630598,3535.1,58.918333,30,5,168,137,31,136.867415,111.612118,25.255297
2496,1610612760,1628983-1629652-1631096-1631114-1641717,4138.7,68.978333,34,6,194,145,49,134.998913,100.901249,34.097664
18,1610612737,1627777-1629027-1629611-1630168-1630249,3405.0,56.750000,32,2,158,138,20,133.638767,116.722467,16.916300
229,1610612739,1628378-1628386-1629636-1630171-1630596,9976.9,166.281667,71,14,458,372,86,132.209404,107.384057,24.825347
520,1610612742,202681-202691-1629023-1629029-1641726,4846.0,80.766667,34,7,222,188,34,131.935617,111.729261,20.206356


,team_id,lineup_key,total_seconds,total_minutes,stints,games,points_for,points_against,net_points,points_for_per_48,points_against_per_48,net_points_per_48
2453,1610612760,1627936-1629652-1630198-1630598-1631096,4307.0,71.783333,34,6,171,121,50,114.344091,80.910146,33.433945
2482,1610612760,1628392-1629026-1630198-1630598-1631114,3516.4,58.606667,32,5,141,102,39,115.481743,83.539984,31.941759
2228,1610612757,203924-1629014-1630166-1630625-1630703,3559.0,59.316667,38,4,126,109,17,101.961225,88.204552,13.756673
1084,1610612747,2544-1626156-1629060-1629216-1629629,3486.9,58.115000,29,5,121,107,14,99.939775,88.376495,11.563280
1465,1610612751,1626156-1627732-1629651-1630533-1630549,4143.6,69.060000,42,3,152,130,22,105.647263,90.356212,15.291051
2403,1610612759,101108-203084-1630170-1631110-1641705,6335.9,105.598333,44,9,236,200,36,107.274420,90.910526,16.363895
3064,1610612765,202699-1627736-1630191-1630595-1631093,5624.7,93.745000,50,5,216,183,33,110.597899,93.700997,16.896901
1694,1610612753,202709-203484-1628371-1628976-1629021,3547.0,59.116667,35,3,125,116,9,101.494220,94.186637,7.307584
140,1610612738,201143-201950-1628369-1628401-1629674,3453.6,57.560000,27,4,125,113,12,104.239055,94.232106,10.006949
1828,1610612754,204456-1626167-1627783-1628396-1628418,4083.0,68.050000,43,4,165,134,31,116.385011,94.518736,21.866275


,team_id,lineup_key,total_seconds,total_minutes,stints,games,points_for,points_against,net_points,points_for_per_48,points_against_per_48,net_points_per_48
1007,1610612746,201587-202695-203992-1626181-1627732,4243.0,70.716667,31,6,204,144,60,138.468065,97.742164,40.725901
2496,1610612760,1628983-1629652-1631096-1631114-1641717,4138.7,68.978333,34,6,194,145,49,134.998913,100.901249,34.097664
2453,1610612760,1627936-1629652-1630198-1630598-1631096,4307.0,71.783333,34,6,171,121,50,114.344091,80.910146,33.433945
2459,1610612760,1628392-1628983-1629026-1629652-1630598,4187.6,69.793333,36,6,202,155,47,138.924444,106.600439,32.324004
2482,1610612760,1628392-1629026-1630198-1630598-1631114,3516.4,58.606667,32,5,141,102,39,115.481743,83.539984,31.941759
647,1610612743,201566-1627750-1629008-1631124-1631128,3534.1,58.901667,27,5,177,143,34,144.240401,116.533205,27.707196
135,1610612738,201143-201950-1627759-1628401-1628436,4229.3,70.488333,30,5,192,153,39,130.745041,104.187454,26.557586
993,1610612746,201587-201935-202695-203992-1626181,11160.1,186.001667,100,11,488,387,101,125.934355,99.870073,26.064283
2430,1610612760,1627936-1628392-1629026-1630198-1630598,3535.1,58.918333,30,5,168,137,31,136.867415,111.612118,25.255297
546,1610612742,202681-202691-203915-203939-1629029,4118.3,68.638333,39,4,174,138,36,121.681276,96.505840,25.175436
